In [ ]:
!pip install --upgrade pip setuptools wheel
!pip install ultralytics opencv-python-headless matplotlib pandas

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from ultralytics import YOLO
from ultralytics import RTDETR

In [ ]:

#Training: 60% Wild + 40% Web

#Test: 60% Web + 40% Wild

print("\nExperiment 60% Wild + 40% Web training\n")

base_path = "/datasetfolderpath/anonymous/lote-animal-mixed-domain-s2"

train_image_dir = f"{base_path}/train/images"
train_label_dir = f"{base_path}/train/labels"
val_image_dir   = f"{base_path}/val/images"
val_label_dir   = f"{base_path}/val/labels"
test_image_dir  = f"{base_path}/test/images"
test_label_dir  = f"{base_path}/test/labels"

yaml_path1 = "/datasetfolderpath/anonymous/s2_rtdetr/working/dataset1.yaml"

run_name = "exp_wild60_web40"

best_model_path = f"/datasetfolderpath/anonymous/s2_rtdetr/working/runs/detect/{run_name}/weights/best.pt"
resume_path     = f"/datasetfolderpath/anonymous/s2_rtdetr/working/runs/detect/{run_name}/weights/last.pt"


In [ ]:
class_names = [
    'Pandas','RedPandas','MartesFlavigula','MacacaThibetana',
    'RhinopithecusRoxellana','HystrixBrachyura','SusScrofa',
    'RusaUnicolor','ElaphodusCephalophus','CapricornisMilneedwardsii',
    'PseudoisNayaur'
]

dataset_yaml = f"""
path: {base_path}
train: train/images
val: val/images
test: test/images
nc: 11
names: {class_names}
"""

with open(yaml_path1, "w") as f:
    f.write(dataset_yaml)


In [ ]:
if os.path.exists(resume_path):
    model = RTDETR(resume_path)
    resume_flag = True
else:
    model = RTDETR("rtdetr-l.pt")
    resume_flag = False

model.train(
    data=yaml_path1,
    epochs=500,
    imgsz=800,
    batch=8,
    device=0,
    workers=4,
    patience=30,
    pretrained=True,
    optimizer='auto',
    weight_decay=0.0005,
    exist_ok=True,
    verbose=True,
    name=run_name,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.001,
    flipud=0.01,
    fliplr=0.5,
    mosaic=0.5,
    mixup=0.1,
    copy_paste=0.1,
    resume=resume_flag
)


In [ ]:
print("\nModel validation on test set!")
print("-" * 20)

model = RTDETR(best_model_path)
test_metrics1 = model.val(data=yaml_path1, split="test", imgsz=800)

print("\nValidation Results:")
print(f"mAP50:    {test_metrics1.box.map50:.4f}")
print(f"mAP50-95: {test_metrics1.box.map:.4f}")
print(f"Precision: {test_metrics1.box.p.mean():.4f}")
print(f"Recall:    {test_metrics1.box.r.mean():.4f}")